# Algorithm: Sampling and Training an RBM (Block Gibbs and CD-k)
In the general Boltzmann machine, Gibbs sampling updates one node at a time: pick a node, compute its conditional probability given every other node, and sample. This is slow because every node is coupled to every other node.

<div>
    <center>
      <img
        src="figs/Fig-Restricted-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="264"
        width="800"
      />
    </center>
</div>

The RBM's bipartite structure changes this entirely. Because there are no connections within a layer, every hidden unit is conditionally independent of every other hidden unit given the visible layer, and vice versa. This means all hidden units can be updated simultaneously in one block, and all visible units in the next. This is called **block Gibbs sampling**.

> __Feedforward and feedback passes__: 
> 
> The feedforward pass computes the hidden layer activations given the visible layer, and the feedback pass computes the visible layer activations given the hidden layer. Because of the RBM's structure, these passes can be computed in parallel across all units in the layer at once.
> 
> The total input to hidden unit $h_{j}$ from the visible layer is $\sum_{i}w_{ij}v_{i} + b_{j}$, which in matrix form is $(\mathbf{W}^{\top}\mathbf{v})_{j} + b_{j}$. The probability that $h_{j}$ is `on` given the current visible state $\mathbf{v}$ follows the same logistic form as the general Boltzmann machine, but now the sum runs only over the visible-to-hidden connections:
> $$
P\!\left(h_{j}=1 \mid \mathbf{v}\right) = \frac{1}{1+\exp\!\left(-2\beta\left((\mathbf{W}^{\top}\mathbf{v})_{j}+b_{j}\right)\right)}.
> $$
> Symmetrically, the total input to visible unit $v_{i}$ from the hidden layer is $(\mathbf{W}\mathbf{h})_{i} + a_{i}$, and the probability that $v_{i}$ is `on` given the current hidden state $\mathbf{h}$ is:
> $$
> P\!\left(v_{i}=1 \mid \mathbf{h}\right) = \frac{1}{1+\exp\!\left(-2\beta\left((\mathbf{W}\mathbf{h})_{i}+a_{i}\right)\right)}.
> $$
> Because all units in a layer are conditionally independent, both of these updates can be computed in parallel across all units in the layer at once, which is the key computational advantage of the RBM structure.

### Sampling from the RBM (Block Gibbs Sampling)
To sample from the RBM, we can run block Gibbs sampling. Starting from an initial visible state $\mathbf{v}^{(0)}$, we alternately sample the hidden layer given the visible layer, and then the visible layer given the hidden layer, for a number of turns $T$. After enough turns, the chain of states $\left\{(\mathbf{v}^{(t)}, \mathbf{h}^{(t)})\right\}_{t=1}^{T}$ will converge to samples from the model distribution $p_\theta(\mathbf{v}, \mathbf{h})$.

**Algorithm (Block Gibbs Sampling)**

__Initialize__: choose parameters $\mathbf{W}$, $\mathbf{a}$, $\mathbf{b}$; an initial visible state $\mathbf{v}^{(0)} \in \{-1, 1\}^{n}$; inverse temperature $\beta > 0$; and the number of turns $T\in\mathbb{Z}_{>0}$.

For each turn $t = 1, 2, \dots, T$:
1. For each hidden unit $h_{j}$, compute $P(h_{j}^{(t)} = 1 \mid \mathbf{v}^{(t-1)})$ using the expression above, then sample $h_{j}^{(t)} \in \{-1, 1\}$ from a Bernoulli distribution with that probability. All hidden units are sampled simultaneously.
2. For each visible unit $v_{i}$, compute $P(v_{i}^{(t)} = 1 \mid \mathbf{h}^{(t)})$ using the expression above, then sample $v_{i}^{(t)} \in \{-1, 1\}$ from a Bernoulli distribution with that probability. All visible units are sampled simultaneously.
3. Store $(\mathbf{v}^{(t)}, \mathbf{h}^{(t)})$ and proceed to the next turn.

After $T$ turns, the chain $\left\{(\mathbf{v}^{(t)}, \mathbf{h}^{(t)})\right\}_{t=1}^{T}$ converges to samples from the model distribution $p_\theta(\mathbf{v}, \mathbf{h})$.

### Contrastive Divergence (CD-$k$)
Running block Gibbs to convergence at every parameter update is still expensive. Hinton's key practical insight, contrastive divergence, is that you do not need the chain to mix fully. Instead, run only $k$ block Gibbs steps (typically $k=1$ works well in practice) starting from the training pattern itself, and use the result as a cheap approximation to the model's distribution. The gradient approximation from the training theory then becomes:

$$
\Delta\mathbf{W} \approx \eta\left(\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{+}-\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{-}\right),
$$

where $\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{+}$ is computed with the visible units clamped to the training pattern (the **positive phase**, what the data says), and $\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{-}$ is computed after $k$ block Gibbs steps starting from that pattern (the **negative phase**, a short approximation of what the model believes).

**Algorithm (CD-$k$)**

__Initialize__: set $\mathbf{W}$, $\mathbf{a}$, $\mathbf{b}$ to small random values; choose learning rate $\eta > 0$, inverse temperature $\beta = 1$, Gibbs depth $k \geq 1$, tolerance $\epsilon > 0$, and maximum epochs $N_{\text{epochs}}$.

For each epoch $e = 1, 2, \dots, N_{\text{epochs}}$, and for each training pattern $\mathbf{x}^{(r)} \in \mathbf{X}$:
1. **Positive phase**: clamp the visible units to the training pattern, $\mathbf{v}^{(0)} = \mathbf{x}^{(r)}$. Compute $p_{j}^{+} = P(h_{j}=1 \mid \mathbf{v}^{(0)})$ for all $j$, and sample $\mathbf{h}^{(0)} \sim P(\mathbf{h} \mid \mathbf{v}^{(0)})$. Form the data-driven correlation $\langle v_{i} h_{j}\rangle^{+} = v_{i}^{(0)}\,p_{j}^{+}$.
2. **Negative phase**: run $k$ block Gibbs steps by alternately sampling $\mathbf{v}^{(t)} \sim P(\mathbf{v} \mid \mathbf{h}^{(t-1)})$ then $\mathbf{h}^{(t)} \sim P(\mathbf{h} \mid \mathbf{v}^{(t)})$ for $t = 1, \dots, k$. Compute $p_{j}^{-} = P(h_{j}=1 \mid \mathbf{v}^{(k)})$ and form $\langle v_{i} h_{j}\rangle^{-} = v_{i}^{(k)}\,p_{j}^{-}$.
3. **Update**: use the difference between the positive and negative phases to update all parameters:
$$
\mathbf{W}\leftarrow\mathbf{W}+\eta\left(\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{+}-\langle\mathbf{v}\mathbf{h}^{\top}\rangle^{-}\right)
$$
$$
\mathbf{a}\leftarrow\mathbf{a}+\eta\left(\langle\mathbf{v}\rangle_{\text{data}}-\langle\mathbf{v}\rangle_{\text{model}}\right),\qquad
\mathbf{b}\leftarrow\mathbf{b}+\eta\left(\langle\mathbf{h}\rangle_{\text{data}}-\langle\mathbf{h}\rangle_{\text{model}}\right).
$$
4. Check stopping criteria. If any criterion is met, stop; otherwise continue.

**Stopping Criteria**

Training is stopped when the parameter updates become negligible, meaning the model has essentially converged. This can be checked in three ways:
- **Parameter change**: $\|\Delta\mathbf{W}\|_{F} < \epsilon$, $\|\Delta\mathbf{a}\|_{2} < \epsilon$, and $\|\Delta\mathbf{b}\|_{2} < \epsilon$.
- **Correlation matching**: the positive and negative phase correlations agree, $\max_{i,j}\left|\langle v_{i}h_{j}\rangle^{+} - \langle v_{i}h_{j}\rangle^{-}\right| < \epsilon$, and similarly for the visible and hidden bias updates.
- **Fixed budget**: stop after $N_{\text{epochs}}$ epochs regardless.

> CD-$k$ trades exactness for speed. The negative phase samples do not come from the true model distribution; they come from a chain that has only taken $k$ steps. In practice, $k=1$ often works well, and the resulting weight updates are a useful (if biased) approximation to the true gradient.

___